# Bayes' Theorem and Inference

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 03.04 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/05_bayes_theorem_and_inference.ipynb)

---

## 🎯 Learning Objective

Derive Bayes' theorem and use it to update beliefs from prior to posterior, culminating in maximum a posteriori (MAP) estimation.

---

## 📐 Theory

Bayes' theorem is the mathematical engine of belief revision: given some new evidence, exactly
how should you update what you believed before seeing it? It's a direct algebraic consequence
of the conditional probability definition from `03.01`, but its implications run through all of
statistics and machine learning.

### Deriving Bayes' theorem

Conditional probability (`03.01`) gives two ways to write the same joint probability:

$$P(A\mid B)P(B) = P(A\cap B) = P(B\mid A)P(A).$$

Dividing through by $P(B)$ gives **Bayes' theorem**:

$$P(A\mid B) = \frac{P(B\mid A)\,P(A)}{P(B)}.$$

That's the entire derivation — no new axioms, just symmetry of $P(A\cap B)$.

### The four named pieces

In inference, we relabel $A\to\theta$ (a parameter or hypothesis) and $B\to D$ (observed data),
and every factor gets a name:

$$\underbrace{P(\theta\mid D)}_{\text{posterior}} = \frac{\overbrace{P(D\mid\theta)}^{\text{likelihood}}\;\overbrace{P(\theta)}^{\text{prior}}}{\underbrace{P(D)}_{\text{evidence}}}.$$

- **Prior** $P(\theta)$: what you believed about $\theta$ *before* seeing data $D$.
- **Likelihood** $P(D\mid\theta)$: how probable the observed data is, *if* $\theta$ were true —
  the same object [Maximum Likelihood Estimation](07_maximum_likelihood_estimation.ipynb)
  (`03.07`) maximizes on its own, with no prior at all.
- **Evidence** $P(D) = \int P(D\mid\theta)P(\theta)\,d\theta$: the normalizing constant that
  makes the posterior integrate to 1 — by the law of total probability (`03.01`), marginalizing
  the joint $P(D,\theta)$ over every possible $\theta$.
- **Posterior** $P(\theta\mid D)$: your updated belief about $\theta$ *after* seeing $D$ —
  strictly more informed than the prior alone, and the object
  [Bayesian Deep Learning](11_bayesian_deep_learning.ipynb) (`03.11`) puts directly on a neural
  network's weights.

### MAP estimation

Since $P(D)$ doesn't depend on $\theta$, maximizing the posterior over $\theta$ only needs the
numerator:

$$\hat{\theta}_{\text{MAP}} = \arg\max_\theta P(\theta\mid D) = \arg\max_\theta P(D\mid\theta)\,P(\theta).$$

**Maximum a posteriori (MAP)** estimation is MLE with a prior bolted on: drop the prior term
(equivalently, make it uniform/flat) and MAP collapses exactly to MLE. Keeping a Gaussian prior
centered at zero turns MAP into L2-regularized MLE — the prior's log becomes a penalty term
that discourages large parameter values, precisely how weight decay is derived in
[Constrained Optimization](../04_Optimization/05_constrained_optimization.ipynb) (`04.05`).

### Sequential updating

Bayes' theorem composes: today's posterior becomes tomorrow's prior. Observing data
$D_1,D_2$ one at a time, $P(\theta\mid D_1,D_2) \propto P(D_2\mid\theta)\,P(\theta\mid D_1)
\propto P(D_2\mid\theta)P(D_1\mid\theta)P(\theta)$ — updating twice sequentially gives the exact
same posterior as updating once on both points jointly (given independence of $D_1,D_2$ given
$\theta$). This is what makes "learning from a stream of data" mathematically coherent rather
than an ad-hoc procedure.

### Conjugate priors

A prior is **conjugate** to a likelihood if the resulting posterior has the same *functional
form* as the prior (just with updated parameters), making updates a closed-form parameter
change instead of a hard integral. The Beta-Bernoulli pair used below is the canonical example,
and [A Gallery of Common Distributions](06_common_distributions_gallery.ipynb) (`03.06`) tours
several more.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta, norm
from sklearn.linear_model import Ridge
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Estimating a coin's bias from flips, one flip at a time: watch the posterior (a Beta
distribution) start as a flat prior and sharpen around the true bias as evidence accumulates.

In [ ]:
# Beta-Bernoulli conjugate updating: start from a flat (uniform) prior on a coin's bias theta,
# and update the posterior one flip at a time using Bayes' theorem in closed form.
rng = np.random.default_rng(0)
true_bias = 0.7
flips = rng.binomial(1, true_bias, size=40)   # 1=heads, 0=tails

# For a Beta(a,b) prior and Bernoulli likelihood, the posterior after k heads / (n-k) tails
# is EXACTLY Beta(a+k, b+n-k) -- conjugacy means no numerical integration is ever needed.
a_prior, b_prior = 1.0, 1.0   # Beta(1,1) = Uniform(0,1): "no opinion" prior
theta_grid = np.linspace(0.001, 0.999, 500)

checkpoints = [0, 1, 5, 10, 40]
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(checkpoints)))
for n_seen, color in zip(checkpoints, colors):
    k_heads = flips[:n_seen].sum()
    a_post, b_post = a_prior + k_heads, b_prior + (n_seen - k_heads)
    ax.plot(theta_grid, beta(a_post, b_post).pdf(theta_grid), color=color, lw=2,
            label=f"n={n_seen} flips seen ({int(k_heads)} heads)")
ax.axvline(true_bias, color="red", ls="--", lw=1.5, label=f"true bias={true_bias}")
ax.set_xlabel(r"$\theta$ (hypothesized coin bias)"); ax.set_ylabel("posterior density")
ax.set_title("Bayes' theorem in action: the posterior sharpens around the truth as data arrives")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

final_k = flips.sum()
a_final, b_final = a_prior + final_k, b_prior + (len(flips) - final_k)
posterior_mean = a_final / (a_final + b_final)
posterior_mode_map = (a_final - 1) / (a_final + b_final - 2)  # MAP estimate: mode of Beta(a,b), a,b>1
print(f"After {len(flips)} flips ({final_k} heads): posterior mean={posterior_mean:.4f}, "
      f"MAP estimate={posterior_mode_map:.4f}, true bias={true_bias}")
print(f"Plain MLE estimate (just the observed frequency) = {final_k/len(flips):.4f}")

## 🐍 Implementation from Scratch

We implement Bayes' theorem directly (grid approximation, no conjugacy shortcut), verify it
matches the closed-form Beta-Bernoulli update, then implement MAP estimation for a Gaussian
mean with a Gaussian prior and confirm it interpolates between the prior mean and the MLE.

In [ ]:
# Bayes' theorem via brute-force grid approximation: NO conjugacy assumed here, just
# numerator = likelihood * prior, then normalize by summing over the grid (discretized evidence).
def bayes_update_grid(theta_grid, prior_pdf_vals, log_likelihood_fn, data):
    log_lik = np.array([sum(log_likelihood_fn(d, theta) for d in data) for theta in theta_grid])
    unnormalized_posterior = np.exp(log_lik - log_lik.max()) * prior_pdf_vals  # shift for stability
    grid_spacing = theta_grid[1] - theta_grid[0]
    evidence = np.sum(unnormalized_posterior) * grid_spacing        # law of total probability, discretized
    return unnormalized_posterior / evidence

theta_grid = np.linspace(0.001, 0.999, 2000)
prior_vals = np.ones_like(theta_grid)   # Beta(1,1) prior, evaluated on the grid
bernoulli_log_lik = lambda d, theta: d * np.log(theta) + (1 - d) * np.log(1 - theta)

rng = np.random.default_rng(0)
true_bias, n_flips = 0.7, 40
flips = rng.binomial(1, true_bias, size=n_flips)

posterior_grid = bayes_update_grid(theta_grid, prior_vals, bernoulli_log_lik, flips)

# Cross-check against the closed-form conjugate Beta posterior from the Visual Intuition cell
k_heads = flips.sum()
posterior_closed_form = beta(1 + k_heads, 1 + n_flips - k_heads).pdf(theta_grid)

print(f"Grid-approximation posterior integrates to: {np.sum(posterior_grid) * (theta_grid[1]-theta_grid[0]):.4f} (expect 1)")
print(f"Max |grid posterior - closed-form Beta posterior| = {np.max(np.abs(posterior_grid - posterior_closed_form)):.4f}")
print("(small and shrinking with a finer grid -- this is discretization error, not a modeling error,")
print(" since the grid method makes no conjugacy assumption at all)")

# MAP estimation for a Gaussian mean, Gaussian prior -- fully closed form, derived by hand:
# maximizing log(likelihood) + log(prior) over mu gives a precision-weighted average of the
# prior mean and the sample mean.
def gaussian_map_mean(data, prior_mean, prior_var, likelihood_var):
    n = len(data)
    sample_mean = data.mean()
    # precision = 1/variance; MAP mean = weighted average of prior mean and sample mean,
    # weighted by their respective precisions (more data => more weight on the sample mean)
    prior_precision, data_precision = 1 / prior_var, n / likelihood_var
    map_mean = (prior_precision * prior_mean + data_precision * sample_mean) / (prior_precision + data_precision)
    return map_mean, sample_mean

rng2 = np.random.default_rng(1)
true_mu, known_sigma2 = 5.0, 4.0
prior_mean, prior_var = 0.0, 1.0   # a prior that's fairly confident mu is near 0 (wrong, on purpose)

print("\n--- MAP estimation of a Gaussian mean, as a function of sample size n ---")
for n in [1, 5, 50, 5000]:
    data = rng2.normal(true_mu, np.sqrt(known_sigma2), size=n)
    map_est, mle_est = gaussian_map_mean(data, prior_mean, prior_var, known_sigma2)
    print(f"  n={n:5d}: MLE (sample mean)={mle_est:6.3f}, MAP={map_est:6.3f}, "
          f"prior mean={prior_mean}, true mu={true_mu}")
print("As n grows, MAP is pulled away from the (wrong) prior and converges to the MLE --")
print("exactly as the theory predicts: more data means the likelihood dominates the prior.")

## 🤖 Why This Matters for AI

**Bayesian learning** reframes training itself as posterior updating: start with a prior over
model weights, observe training data, compute a posterior. **MAP estimation** is the practical
entry point — it's literally what L2-regularized maximum likelihood *is*: a Gaussian prior on
the weights, in log-space, becomes exactly the weight-decay penalty term added to a loss
function. Below, we make that equivalence concrete: fitting a linear model by minimizing
squared error plus an L2 penalty produces the *exact same* weights as computing the MAP estimate
under a Gaussian likelihood and a Gaussian prior on the weights.

---

In [ ]:
# Showing L2-regularized least squares IS MAP estimation with a Gaussian prior on the weights.
# Model: y = X @ w + noise, noise ~ N(0, sigma^2).  Prior: w ~ N(0, tau^2 * I).
rng = np.random.default_rng(0)
n_samples, n_features = 200, 5
X = rng.normal(size=(n_samples, n_features))
true_w = np.array([2.0, -1.0, 0.5, 0.0, 3.0])
sigma2 = 4.0   # noise variance
y = X @ true_w + rng.normal(scale=np.sqrt(sigma2), size=n_samples)

tau2 = 1.0     # prior variance on each weight
lam = sigma2 / tau2   # the ridge penalty strength implied by this exact prior (derived below)

# Route 1: MAP estimate in closed form -- maximizing log p(w|D) directly gives ridge regression's
# normal equations: w_MAP = (X^T X + lambda*I)^-1 X^T y, with lambda = sigma^2 / tau^2
w_map = np.linalg.solve(X.T @ X + lam * np.eye(n_features), X.T @ y)

# Route 2: explicit ridge regression -- minimize ||y - Xw||^2 + lambda*||w||^2 via sklearn,
# with NO reference to "priors" or "posteriors" anywhere in its own formulation
ridge = Ridge(alpha=lam, fit_intercept=False)
ridge.fit(X, y)
w_ridge = ridge.coef_

print("w_MAP (derived from Bayes' theorem + Gaussian prior):", np.round(w_map, 4))
print("w_ridge (sklearn Ridge, penalty lambda=sigma^2/tau^2):", np.round(w_ridge, 4))
print(f"Max abs difference: {np.max(np.abs(w_map - w_ridge)):.2e}  (should be ~0: these are the SAME estimator)")

# Route 3, as a sanity anchor: plain MLE (lambda=0, i.e. an infinitely flat/uninformative prior)
w_mle = np.linalg.solve(X.T @ X, X.T @ y)
print(f"\nPlain MLE (no prior, lambda=0):        {np.round(w_mle, 4)}")
print(f"True weights used to generate the data: {true_w}")
print(f"||w_MAP - true_w|| = {np.linalg.norm(w_map - true_w):.4f}   "
      f"||w_MLE - true_w|| = {np.linalg.norm(w_mle - true_w):.4f}")
print("The prior pulls small/noisy coefficients toward zero, trading a little bias for lower")
print("variance -- exactly the bias-variance tradeoff regularization is used for in practice.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A disease affects 1% of a population. A test is 99% sensitive ($P(\text{positive}\mid
   \text{disease})=0.99$) and 95% specific ($P(\text{negative}\mid\text{no disease})=0.95$).
   By hand, use Bayes' theorem to compute $P(\text{disease}\mid\text{positive})$. As a sanity
   check, confirm your posterior is a valid probability (between 0 and 1), and verify with code.

<details>
<summary>💡 Solution</summary>

By the law of total probability, $P(\text{positive}) = P(\text{pos}\mid\text{disease})P(\text{disease})
+ P(\text{pos}\mid\text{no disease})P(\text{no disease}) = 0.99\times 0.01 + 0.05\times 0.99 =
0.0099+0.0495=0.0594$. Bayes' theorem: $P(\text{disease}\mid\text{positive}) =
\frac{0.99\times 0.01}{0.0594} = \frac{0.0099}{0.0594} \approx 0.1667$. This is a valid
probability ($0\le 0.1667\le 1$), and it is dramatically lower than the test's own $99\%$
sensitivity — because the disease is rare, most positives come from the much larger healthy
population's $5\%$ false-positive rate, not from the small diseased population's true positives.

</details>

### 💭 UNDERSTAND
2. Two people analyze the exact same coin-flip data with two different priors: person A uses a
   flat `Beta(1,1)` prior, person B uses a confident `Beta(50,50)` prior (strongly believes the
   coin is fair). After observing the same 40 flips (27 heads), explain — without computing the
   exact posteriors — why person B's posterior mean will end up closer to $0.5$ than person A's,
   and identify what would make the two posteriors converge despite the different priors.

<details>
<summary>💡 Solution</summary>

A `Beta(a,b)` prior's "strength" is its *effective sample size* $a+b$ — `Beta(1,1)` has strength
$2$ (barely more than one imaginary prior observation), while `Beta(50,50)` has strength $100$,
as if person B had already privately seen 50 heads and 50 tails before any real data arrived.
Since the posterior parameters are (prior $+$ observed counts), the same 40 real observations
make up a much larger *share* of the total evidence for person A ($40$ new vs. $2$ prior) than
for person B ($40$ new vs. $100$ prior) — so B's posterior stays pulled toward the prior's
$0.5$ far more than A's. The two posteriors converge as the amount of *real* data grows large
enough to swamp even B's strong prior — with enough flips, both priors' fixed initial
contribution becomes negligible relative to the ever-growing observed counts, and both
posteriors concentrate on the same true bias.

</details>

### ✏️ DERIVE
3. The notebook's `gaussian_map_mean` function computes the MAP estimate of a Gaussian's mean
   (known variance $\sigma^2$, Gaussian prior $\mu\sim N(\mu_0,\tau^2)$) as a precision-weighted
   average, without deriving it. Derive it: write the log-posterior
   $\log p(\mu\mid D) \propto \log p(D\mid\mu) + \log p(\mu)$ using the Gaussian likelihood and
   Gaussian prior, differentiate with respect to $\mu$, set the derivative to zero, and solve
   for $\hat\mu_{\text{MAP}}$.

<details>
<summary>💡 Solution outline</summary>

Up to additive constants not involving $\mu$: $\log p(D\mid\mu) = -\frac{1}{2\sigma^2}\sum_i
(x_i-\mu)^2 = -\frac{1}{2\sigma^2}\big(n\mu^2 - 2\mu\sum_i x_i\big) + \text{const} =
-\frac{n}{2\sigma^2}\mu^2 + \frac{n\bar{x}}{\sigma^2}\mu + \text{const}$, and $\log p(\mu) =
-\frac{1}{2\tau^2}(\mu-\mu_0)^2$. Sum and differentiate with respect to $\mu$:
$$\frac{d}{d\mu}\log p(\mu\mid D) = -\frac{n\mu}{\sigma^2}+\frac{n\bar x}{\sigma^2} -
\frac{\mu-\mu_0}{\tau^2}.$$
Setting this to zero and solving for $\mu$:
$$\mu\left(\frac{n}{\sigma^2}+\frac{1}{\tau^2}\right) = \frac{n\bar x}{\sigma^2}+\frac{\mu_0}{\tau^2}
\implies \hat\mu_{\text{MAP}} = \frac{\frac{1}{\tau^2}\mu_0 + \frac{n}{\sigma^2}\bar x}{\frac{1}{\tau^2}+\frac{n}{\sigma^2}},$$
exactly a precision-weighted ($1/\tau^2$ = prior precision, $n/\sigma^2$ = data precision)
average of the prior mean $\mu_0$ and the sample mean $\bar x$ — precisely the formula
`gaussian_map_mean` implements. As $n\to\infty$, the data-precision term dominates and
$\hat\mu_{\text{MAP}}\to\bar x$, the MLE — matching the notebook's own numerical observation.

</details>

### 🐍 IMPLEMENT
4. In the Beta-Bernoulli grid-approximation code, change the prior to `beta(10, 10).pdf(...)`
   (a prior that already believes the coin is close to fair) instead of the flat `Beta(1,1)`
   prior, and rerun `bayes_update_grid` with the same 40 flips. Check your grid-based posterior
   against the closed-form conjugate update `beta(10 + k_heads, 10 + n_flips - k_heads)`.

<details>
<summary>✅ How to know you're right</summary>

With `prior_vals = beta(10, 10).pdf(theta_grid)` substituted into `bayes_update_grid`, the
resulting `posterior_grid` should still integrate to $1$ (check
`np.sum(posterior_grid) * (theta_grid[1]-theta_grid[0])` is close to `1.0`), and the maximum
absolute difference between `posterior_grid` and the closed-form
`beta(10 + k_heads, 10 + n_flips - k_heads).pdf(theta_grid)` should be small (comparable in
magnitude to the discrepancy the notebook already reports for the flat-prior case) and shrink
further with a finer grid — since, exactly as with the flat prior, the grid method makes no
conjugacy assumption and should agree with the closed form up to discretization error only.

</details>

### 🤖 APPLY
5. In the L2-regularization-as-MAP cell, sweep the prior variance `tau2` over
   `[0.01, 0.1, 1.0, 10.0, 100.0]` (equivalently sweeping the ridge penalty
   $\lambda=\sigma^2/\tau^2$ from strong to weak) and compute $\|w_{\text{MAP}}-w_{\text{true}}\|$
   at each value.

<details>
<summary>✅ What you should observe</summary>

The error should be U-shaped in $\log(\tau^2)$: large at `tau2=0.01` (a very strong, overly
restrictive prior pulling weights hard toward zero — error around $2.6$ in a representative
run), decreasing through `tau2=0.1` and `tau2=1.0`, reaching its smallest value somewhere
between `tau2=0.1` and `tau2=10` (in a representative run, near `tau2≈0.3`–`1`, error dropping
to roughly $0.2$–$0.25$), and then leveling off toward the plain-MLE error (around $0.33$ in the
same run) as `tau2` keeps growing and the prior becomes negligible. The fact that both
extremes — too strong a prior and too weak a prior — give worse recovery of `true_w` than some
intermediate value is exactly the bias-variance tradeoff regularization strength controls, and
is why $\lambda$ (equivalently the assumed prior variance) is tuned via validation data rather
than fixed a priori.

</details>

### 🚀 CHALLENGE
6. Sequential updating means $P(\theta\mid D_1,D_2)\propto P(D_2\mid\theta)P(\theta\mid D_1)$ —
   updating on $D_1$ then $D_2$ should give the identical posterior as updating on both at once
   (given $D_1,D_2$ conditionally independent given $\theta$). Verify this computationally for
   the Beta-Bernoulli setup: split the notebook's 40 flips into two batches (say, flips
   $0$–$19$ and flips $20$–$39$), update a `Beta(1,1)` prior on the first batch to get an
   intermediate posterior, use that posterior as the prior for the second batch, and confirm the
   final result matches updating on all 40 flips at once, for *any* split point you choose (try
   at least two different split points, e.g. 20/20 and 5/35). Then investigate: does the order in
   which you feed the two batches in matter? Explain your finding in terms of what
   the Beta-Bernoulli posterior parameters $(a+k, b+n-k)$ actually depend on.

<details>
<summary>🔍 What a strong answer covers</summary>

For any split point, sequentially updating `Beta(1,1)` on the first batch's $k_1$ heads out of
$n_1$ flips gives `Beta(1+k_1, 1+n_1-k_1)`; using that as the prior for the second batch's $k_2$
heads out of $n_2$ flips gives `Beta(1+k_1+k_2, 1+n_1-k_1+n_2-k_2)` = `Beta(1+k, 1+n-k)` where
$k=k_1+k_2$, $n=n_1+n_2$ — algebraically identical to updating on all $n$ flips at once,
regardless of the split point, since Beta-Bernoulli conjugate updating only ever adds counts. A
strong answer verifies this numerically for at least two split points and gets an exactly
matching (up to floating point) posterior both times. On order: swapping which batch is fed
first does not change the final posterior either, because the final parameters
$(1+k_1+k_2,\ 1+n-k_1-k_2)$ only depend on the *totals* $k=k_1+k_2$ and $n=n_1+n_2$, not on the
order the individual counts were accumulated in — addition is commutative. A strong answer
connects this to why "online" / streaming Bayesian updating is mathematically sound at all: it
doesn't matter whether data arrives in one batch or many small ones, or in what order, as long
as every point is eventually incorporated exactly once.

</details>

---

## 📚 Further Reading
- McElreath, *Statistical Rethinking*, Ch. 2 (Small Worlds and Large Worlds)
- Murphy, *Probabilistic Machine Learning: An Introduction*, Ch. 4 (Statistics) — MAP estimation and conjugate priors
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 1.2.3–1.2.6 (Bayesian Probabilities)

---

*Next notebook: [A Gallery of Common Distributions](06_common_distributions_gallery.ipynb)*